In [ ]:
#import argparse
import json
from pathlib import Path
import gymnasium as gym
import random
import argparse
import random

from config_dqn import DQNConfig
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import os


In [ ]:
def make_env(env_id: str, seed: int) -> gym.Env:
    env = gym.make(env_id)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    return Monitor(env)

def test_hyperparameter(env_id: str =  'CartPole-v1',
                        seed: int | None = None, 
                        timesteps_per_round: int = 1000,
                        max_rounds: int = 2500,
                        reward_stop_count: int = 10,
                        min_stop_value: int = 499,
                        evaluation_episodes: int = 10,
                        **kwargs) -> dict:

    #timesteps_per_round: the number of training steps you learn per round.  You check the reward after each round to measure if the model is trained
    #max_rounds: Stop the experiment after this stage.  Usually use the number of rounds necessary to train the default.
    #reward_stop_count: This is to check how sustained your training is.  It requires your model to sustain the min_stop_value over several training rounds.
    #min_stop_value: This is the value for the reward that is your objective. The maximum for the cartpole environment is 500, meaning it is vertical for 500 steps.
    #evaluation_episodes: the evaluate_policy function tests the model over this number of episodes and returns the mean reward.
    #kwargs is a dictionary with the keys/values as the DQN hyperparameters
    
    #Environment    
    env = make_env(env_id, seed)

    #Construct model
    config = DQNConfig.from_dict(kwargs["kwargs"])
    tensorboard_log = None
    params = config.to_kwargs(seed=seed)
    model = DQN("MlpPolicy", env, **params)

    #Train the model until you reach your in terms of  
    rewardList = []
    round_number=0
    max_reward_count = 0
    while (round_number < max_rounds) and (max_reward_count < reward_stop_count):
        model.learn(total_timesteps=timesteps_per_round)
        mean_reward, std_reward = evaluate_policy(
            model,
            model.get_env(),
            n_eval_episodes=evaluation_episodes,
            deterministic=True,
        )
        if round_number % 100 == 0:
            print(round_number)
            print(mean_reward)
        rewardList.append(mean_reward)
        if mean_reward > min_stop_value:
            max_reward_count = max_reward_count + 1
        else:
            max_reward_count = 0
        round_number = round_number+1
    
    reward_dct = {"training_rounds": round_number,
                    "timestamps_per_round": timesteps_per_round,
                    "final_reward": rewardList[-1],
                    "reward_data": rewardList}
    
    return({**params, **reward_dct})

In [ ]:
#These are the differesnt values that we would like to check.
env_id = 'CartPole-v1'
varList = {"learning_rate": [2e-4,5e-5],
    "buffer_size": [2000000, 500000],
    "learning_starts": [90, 110],
    "batch_size": [25, 40],
    "tau": [.9, .95],
    "gamma": [.98, .97],
    "train_freq": [(2, "episode"),(10, "step")],
    "gradient_steps": [2,-1],
    "n_steps": [2,3],
    "target_update_interval": [9000, 11000],
    "exploration_fraction": [0.08, 0.12],
    "exploration_initial_eps": [0.95,0.99],
    "exploration_final_eps": [.045, .055],
    "max_grad_norm": [9,11]}

In [ ]:
'''1 dim grid search'''
resultList = []
max_rounds = 1500
reward_stop_count = 10
min_stop_value= 499
outputPath = "outputs/"

for var_key in varList:
    valList = varList[var_key]
    seed = random.randint(0,10000)
    for j in range(len(valList)):
        val = valList[j]
        filename = outputPath + 'trial_' + var_key + "_" + str(j) +'.json'
        param = {var_key: val}
        output = test_hyperparameter(env_id='CartPole-v1',
                                    seed=seed,
                                    max_rounds=max_rounds,
                                    kwargs=param)
        with open(filename, 'w') as f:
            json.dump(output, f)

In [ ]:
import random
'''RANDOM grid search'''
resultList = []
max_rounds = 10
outputPath = "outputs/"

trial_length = 25
for trial_num in range(trial_length):
    print("Trial " + str(trial_num))
    seed = random.randint(0,10000)
    
    #These are the differesnt random values that we would like to check.
    param = {"learning_rate": max(1e-4,random.gauss(2e-4,1e-4)),
        "buffer_size": max(0, int(random.gauss(2000000,500000))),
        "batch_size": max(10,int(random.gauss(32,6))),
        "tau": min(.995, max(0.9, int(random.gauss(0.95,.2)))),
        "gamma": min(.995, max(0.975, int(random.gauss(0.985,.005)))),
        "gradient_steps": [1,2,3,4,-1][random.randint(0,4)],
        "exploration_fraction": max(0.01,random.gauss(0.1,0.02)),
        "exploration_initial_eps": min(1.0,random.gauss(1.0,0.02))}

    #Write the results
    filename = outputPath + 'random_trial_'+ str(trial_num) +'.json'
    output =test_hyperparameter(env_id='CartPole-v1',
                                    seed=seed,
                                    max_rounds=max_rounds,
                                    kwargs=param)
    with open(filename, 'w') as f:
        json.dump(output, f)


In [ ]:

import matplotlib.pyplot as plt
#VISUAL COMPARISON OF TRAINING TIMES
fileList = ["trial_defaultB.json", "random_trial8.json"]
rewardList = []
for filename in fileList:
    with open("outputs/" + filename) as f:
        data = json.load(f)
    rewardList.append(data["reward_data"])
plt.scatter([i for i in range(len(rewardList[0]))], rewardList[0])
plt.scatter([i for i in range(len(rewardList[1]))], rewardList[1])

In [ ]:
import json
import pandas as pd

#LOOK AT EXISTING FILES AND FIND SHORTEST TRAINING TIMES
#This will come from a sorted pandas dataframe

base_file =  'trial_defaultB.json'
fileList = ['greedy_trial0.json',
            'greedy_trial1.json',
            'greedy_trial2.json',
            'greedy_trial3.json',
            'greedy_trial4.json',
            'greedy_trial5.json',
            'greedy_trial6.json',
            'greedy_trial7.json',
            'random_trial8.json',
 'random_trial1.json',
 'random_trial10.json',
 'random_trial11.json',
 'random_trial12.json',
 'random_trial13.json',
 'random_trial14.json',
 'random_trial15.json',
 'random_trial16.json',
 'random_trial17.json',
 'random_trial18.json',
 'random_trial19.json',
 'random_trial2.json',
 'random_trial20.json',
 'random_trial21.json',
 'random_trial22.json',
 'random_trial23.json',
 'random_trial24.json',
 'random_trial25.json',
 'random_trial26.json',
 'random_trial27.json',
 'random_trial28.json',
 'random_trial29.json',
 'random_trial3.json',
 'random_trial30.json',
 'random_trial4.json',
 'random_trial5.json',
 'random_trial6.json',
 'random_trial7.json',
 'random_trial8.json',
 'random_trial9.json',
 'trial_batch_size_0.json',
 'trial_batch_size_1.json',
 'trial_buffer_size_0.json',
 'trial_buffer_size_1.json',
 'trial_exploration_final_eps_0.json',
 'trial_exploration_final_eps_1.json',
 'trial_exploration_fraction_0.json',
 'trial_exploration_fraction_1.json',
 'trial_exploration_initial_eps_0.json',
 'trial_exploration_initial_eps_1.json',
 'trial_gamma_0.json',
 'trial_gamma_1.json',
 'trial_gradient_steps_0.json',
 'trial_gradient_steps_1.json',
 'trial_learning_rate_0.json',
 'trial_learning_rate_1.json',
 'trial_learning_starts_0.json',
 'trial_learning_starts_1.json',
 'trial_max_grad_norm_0.json',
 'trial_max_grad_norm_1.json',
 'trial_n_steps_0.json',
 'trial_n_steps_1.json',
 'trial_target_update_interval_0.json',
 'trial_target_update_interval_1.json',
 'trial_tau_0.json',
 'trial_tau_1.json',
 'trial_train_freq_0.json',
 'trial_train_freq_1.json']

with open("outputs/" + base_file) as f:
    d = json.load(f)

trList = []
for filename in fileList:
    with open("outputs/" + filename) as f:
        data = json.load(f)
    trList.append(data["training_rounds"])

df = pd.DataFrame({"file": fileList,
                   "train_steps": trList
                  }).sort_values("train_steps")

In [ ]:
#Run another test with a new seed or different evaluation parameters
i=0
filename = "random_trial8.json"
print(filename)
with open("outputs/" + filename) as f:
    data = json.load(f)
test_hyperparameter(env_id =  'CartPole-v1',
                        seed = random.randint(0,1000000), 
                        timesteps_per_round  = 1000,
                        max_rounds = 1500,
                        reward_stop_count=1,
                        min_stop_value = 499,
                        evaluation_episodes = 10,
                        kwargs = data)

In [ ]:
import numpy as np
x = [np.float64(10.5),
  np.float64(9.4),
  np.float64(9.1),
  np.float64(9.2),
  np.float64(12.9),
  np.float64(9.4),
  np.float64(9.9),
  np.float64(11.1),
  np.float64(9.2),
  np.float64(9.3),
  np.float64(9.2),
  np.float64(9.4),
  np.float64(41.6),
  np.float64(9.3),
  np.float64(42.0),
  np.float64(55.1),
  np.float64(35.4),
  np.float64(54.0),
  np.float64(50.4),
  np.float64(10.7),
  np.float64(9.9),
  np.float64(226.1),
  np.float64(10.7),
  np.float64(11.0),
  np.float64(10.5),
  np.float64(32.3),
  np.float64(399.2),
  np.float64(9.0),
  np.float64(12.2),
  np.float64(11.2),
  np.float64(13.5),
  np.float64(21.6),
  np.float64(39.7),
  np.float64(166.9),
  np.float64(122.2),
  np.float64(88.0),
  np.float64(105.7),
  np.float64(74.1),
  np.float64(51.9),
  np.float64(54.4),
  np.float64(143.1),
  np.float64(93.9),
  np.float64(122.8),
  np.float64(108.0),
  np.float64(107.4),
  np.float64(111.5),
  np.float64(92.6),
  np.float64(108.6),
  np.float64(97.7),
  np.float64(93.7),
  np.float64(123.3),
  np.float64(138.8),
  np.float64(133.9),
  np.float64(142.1),
  np.float64(135.6),
  np.float64(135.3),
  np.float64(124.6),
  np.float64(126.8),
  np.float64(107.8),
  np.float64(118.7),
  np.float64(176.8),
  np.float64(160.4),
  np.float64(189.3),
  np.float64(221.1),
  np.float64(139.0),
  np.float64(150.7),
  np.float64(139.0),
  np.float64(134.6),
  np.float64(129.2),
  np.float64(134.6),
  np.float64(168.9),
  np.float64(138.5),
  np.float64(165.6),
  np.float64(157.2),
  np.float64(192.7),
  np.float64(224.4),
  np.float64(167.3),
  np.float64(144.1),
  np.float64(136.8),
  np.float64(165.5),
  np.float64(149.7),
  np.float64(149.5),
  np.float64(168.8),
  np.float64(148.4),
  np.float64(166.4),
  np.float64(151.3),
  np.float64(140.8),
  np.float64(152.9),
  np.float64(182.3),
  np.float64(152.8),
  np.float64(139.6),
  np.float64(175.5),
  np.float64(172.9),
  np.float64(135.3),
  np.float64(178.2),
  np.float64(141.7),
  np.float64(163.1),
  np.float64(140.1),
  np.float64(136.2),
  np.float64(137.4),
  np.float64(151.8),
  np.float64(129.7),
  np.float64(145.0),
  np.float64(128.7),
  np.float64(184.3),
  np.float64(158.2),
  np.float64(127.5),
  np.float64(184.8),
  np.float64(127.8),
  np.float64(136.0),
  np.float64(135.3),
  np.float64(122.7),
  np.float64(127.3),
  np.float64(119.3),
  np.float64(112.0),
  np.float64(124.0),
  np.float64(118.3),
  np.float64(115.1),
  np.float64(115.3),
  np.float64(84.5),
  np.float64(109.3),
  np.float64(112.8),
  np.float64(114.3),
  np.float64(111.8),
  np.float64(111.4),
  np.float64(95.5),
  np.float64(114.6),
  np.float64(108.1),
  np.float64(109.0),
  np.float64(81.6),
  np.float64(107.0),
  np.float64(111.1),
  np.float64(108.9),
  np.float64(109.4),
  np.float64(111.1),
  np.float64(104.9),
  np.float64(108.5),
  np.float64(115.5),
  np.float64(108.7),
  np.float64(116.3),
  np.float64(109.4),
  np.float64(102.1),
  np.float64(102.7),
  np.float64(103.3),
  np.float64(105.2),
  np.float64(106.8),
  np.float64(106.7),
  np.float64(104.5),
  np.float64(103.5),
  np.float64(79.5),
  np.float64(68.0),
  np.float64(137.4),
  np.float64(107.5),
  np.float64(107.3),
  np.float64(60.7),
  np.float64(110.9),
  np.float64(110.1),
  np.float64(77.8),
  np.float64(106.4),
  np.float64(106.1),
  np.float64(108.2),
  np.float64(123.4),
  np.float64(98.0),
  np.float64(126.8),
  np.float64(61.0),
  np.float64(102.8),
  np.float64(137.2),
  np.float64(52.6),
  np.float64(130.3),
  np.float64(30.0),
  np.float64(139.4),
  np.float64(111.0),
  np.float64(92.6),
  np.float64(100.4),
  np.float64(118.4),
  np.float64(184.2),
  np.float64(129.7),
  np.float64(121.7),
  np.float64(126.7),
  np.float64(56.9),
  np.float64(119.6),
  np.float64(94.1),
  np.float64(109.6),
  np.float64(156.2),
  np.float64(129.5),
  np.float64(175.6),
  np.float64(149.6),
  np.float64(108.6),
  np.float64(170.4),
  np.float64(39.3),
  np.float64(198.8),
  np.float64(186.9),
  np.float64(95.7),
  np.float64(132.7),
  np.float64(127.4),
  np.float64(57.0),
  np.float64(87.9),
  np.float64(124.0),
  np.float64(63.0),
  np.float64(34.4),
  np.float64(171.6),
  np.float64(215.2),
  np.float64(215.1),
  np.float64(51.3),
  np.float64(178.9),
  np.float64(220.0),
  np.float64(224.0),
  np.float64(254.8),
  np.float64(70.2),
  np.float64(28.9),
  np.float64(85.8),
  np.float64(286.9),
  np.float64(150.0),
  np.float64(239.2),
  np.float64(99.4),
  np.float64(53.8),
  np.float64(73.7),
  np.float64(251.5),
  np.float64(86.5),
  np.float64(64.0),
  np.float64(272.3),
  np.float64(125.0),
  np.float64(135.7),
  np.float64(125.0),
  np.float64(125.0),
  np.float64(47.5),
  np.float64(80.3),
  np.float64(56.8),
  np.float64(190.7),
  np.float64(51.4),
  np.float64(271.3),
  np.float64(124.6),
  np.float64(92.9),
  np.float64(137.8),
  np.float64(120.3),
  np.float64(191.9),
  np.float64(66.1),
  np.float64(192.4),
  np.float64(100.0),
  np.float64(50.7),
  np.float64(79.4),
  np.float64(134.4),
  np.float64(122.0),
  np.float64(135.2),
  np.float64(269.9),
  np.float64(378.9),
  np.float64(112.9),
  np.float64(145.6),
  np.float64(108.0),
  np.float64(168.2),
  np.float64(183.3),
  np.float64(126.2),
  np.float64(96.8),
  np.float64(117.5),
  np.float64(91.0),
  np.float64(146.7),
  np.float64(164.0),
  np.float64(277.3),
  np.float64(164.7),
  np.float64(163.1),
  np.float64(132.2),
  np.float64(94.5),
  np.float64(95.0),
  np.float64(275.4),
  np.float64(185.8),
  np.float64(105.6),
  np.float64(92.9),
  np.float64(140.6),
  np.float64(82.6),
  np.float64(106.0),
  np.float64(154.2),
  np.float64(69.4),
  np.float64(120.0),
  np.float64(134.3),
  np.float64(145.0),
  np.float64(72.2),
  np.float64(220.1),
  np.float64(90.1),
  np.float64(162.7),
  np.float64(74.3),
  np.float64(85.2),
  np.float64(101.8),
  np.float64(102.2),
  np.float64(148.1),
  np.float64(92.2),
  np.float64(96.8),
  np.float64(174.4),
  np.float64(99.9),
  np.float64(126.6),
  np.float64(89.8),
  np.float64(100.9),
  np.float64(96.6),
  np.float64(94.8),
  np.float64(98.2),
  np.float64(97.2),
  np.float64(240.9),
  np.float64(108.4),
  np.float64(97.7),
  np.float64(103.7),
  np.float64(97.6),
  np.float64(108.3),
  np.float64(94.7),
  np.float64(99.8),
  np.float64(93.3),
  np.float64(95.9),
  np.float64(97.3),
  np.float64(91.0),
  np.float64(174.8),
  np.float64(97.7),
  np.float64(91.6),
  np.float64(267.0),
  np.float64(167.5),
  np.float64(96.0),
  np.float64(317.8),
  np.float64(92.5),
  np.float64(94.2),
  np.float64(99.9),
  np.float64(82.4),
  np.float64(94.1),
  np.float64(92.5),
  np.float64(102.0),
  np.float64(95.9),
  np.float64(388.3),
  np.float64(115.4),
  np.float64(89.6),
  np.float64(133.6),
  np.float64(138.2),
  np.float64(96.2),
  np.float64(97.9),
  np.float64(93.2),
  np.float64(140.5),
  np.float64(93.8),
  np.float64(112.4),
  np.float64(107.9),
  np.float64(259.1),
  np.float64(347.6),
  np.float64(119.8),
  np.float64(126.4),
  np.float64(151.0),
  np.float64(94.2),
  np.float64(133.7),
  np.float64(402.3),
  np.float64(117.4),
  np.float64(432.0),
  np.float64(101.2),
  np.float64(108.3),
  np.float64(123.9),
  np.float64(129.4),
  np.float64(202.7),
  np.float64(99.9),
  np.float64(132.4),
  np.float64(120.3),
  np.float64(105.3),
  np.float64(92.1),
  np.float64(102.1),
  np.float64(98.0),
  np.float64(104.5),
  np.float64(104.1),
  np.float64(220.2),
  np.float64(96.6),
  np.float64(107.3),
  np.float64(132.3),
  np.float64(202.9),
  np.float64(133.5),
  np.float64(109.2),
  np.float64(379.4),
  np.float64(82.8),
  np.float64(112.8),
  np.float64(109.5),
  np.float64(118.4),
  np.float64(194.8),
  np.float64(431.4),
  np.float64(115.8),
  np.float64(127.1),
  np.float64(223.7),
  np.float64(171.4),
  np.float64(101.2),
  np.float64(140.5),
  np.float64(385.3),
  np.float64(86.8),
  np.float64(396.8),
  np.float64(188.8),
  np.float64(242.2),
  np.float64(170.7),
  np.float64(227.5),
  np.float64(165.1),
  np.float64(116.3),
  np.float64(120.5),
  np.float64(208.8),
  np.float64(140.6),
  np.float64(142.9),
  np.float64(394.7),
  np.float64(309.1),
  np.float64(183.3),
  np.float64(253.7),
  np.float64(264.0),
  np.float64(352.0),
  np.float64(377.5),
  np.float64(193.2),
  np.float64(299.4),
  np.float64(99.9),
  np.float64(262.3),
  np.float64(149.6),
  np.float64(179.8),
  np.float64(171.7),
  np.float64(142.6),
  np.float64(136.9),
  np.float64(250.6),
  np.float64(149.7),
  np.float64(108.3),
  np.float64(154.5),
  np.float64(157.5),
  np.float64(150.9),
  np.float64(383.9),
  np.float64(170.6),
  np.float64(288.5),
  np.float64(200.1),
  np.float64(138.3),
  np.float64(115.2),
  np.float64(209.5),
  np.float64(145.2),
  np.float64(100.7),
  np.float64(137.0),
  np.float64(116.6),
  np.float64(284.5),
  np.float64(184.1),
  np.float64(325.4),
  np.float64(163.8),
  np.float64(107.4),
  np.float64(111.6),
  np.float64(254.8),
  np.float64(105.3),
  np.float64(187.1),
  np.float64(165.2),
  np.float64(167.2),
  np.float64(107.5),
  np.float64(209.4),
  np.float64(202.1),
  np.float64(116.7),
  np.float64(281.1),
  np.float64(122.6),
  np.float64(117.9),
  np.float64(112.2),
  np.float64(132.9),
  np.float64(125.9),
  np.float64(261.7),
  np.float64(158.1),
  np.float64(126.1),
  np.float64(188.4),
  np.float64(147.9),
  np.float64(189.4),
  np.float64(181.4),
  np.float64(136.2),
  np.float64(155.1),
  np.float64(154.3),
  np.float64(203.2),
  np.float64(143.9),
  np.float64(141.0),
  np.float64(170.7),
  np.float64(282.3),
  np.float64(121.0),
  np.float64(143.8),
  np.float64(159.5),
  np.float64(133.8),
  np.float64(147.7),
  np.float64(186.2),
  np.float64(127.7),
  np.float64(249.6),
  np.float64(131.5),
  np.float64(160.3),
  np.float64(120.1),
  np.float64(146.5),
  np.float64(236.8),
  np.float64(161.8),
  np.float64(208.6),
  np.float64(120.8),
  np.float64(366.2),
  np.float64(105.6),
  np.float64(148.9),
  np.float64(126.4),
  np.float64(162.0),
  np.float64(253.1),
  np.float64(124.1),
  np.float64(181.4),
  np.float64(110.0),
  np.float64(104.7),
  np.float64(159.2),
  np.float64(115.6),
  np.float64(127.6),
  np.float64(225.5),
  np.float64(475.9),
  np.float64(129.0),
  np.float64(200.8),
  np.float64(143.0),
  np.float64(143.1),
  np.float64(121.7),
  np.float64(105.3),
  np.float64(164.3),
  np.float64(237.4),
  np.float64(104.5),
  np.float64(124.7),
  np.float64(112.4),
  np.float64(128.0),
  np.float64(103.8),
  np.float64(457.1),
  np.float64(228.9),
  np.float64(122.7),
  np.float64(264.6),
  np.float64(130.1),
  np.float64(449.4),
  np.float64(150.3),
  np.float64(341.9),
  np.float64(91.1),
  np.float64(119.7),
  np.float64(113.9),
  np.float64(126.5),
  np.float64(108.7),
  np.float64(97.3),
  np.float64(95.4),
  np.float64(113.6),
  np.float64(110.6),
  np.float64(110.3),
  np.float64(140.1),
  np.float64(106.6),
  np.float64(144.2),
  np.float64(98.7),
  np.float64(105.9),
  np.float64(400.1),
  np.float64(378.6),
  np.float64(111.0),
  np.float64(123.8),
  np.float64(111.0),
  np.float64(158.0),
  np.float64(111.9),
  np.float64(166.7),
  np.float64(102.3),
  np.float64(120.6),
  np.float64(421.4),
  np.float64(116.5),
  np.float64(115.3),
  np.float64(110.9),
  np.float64(111.5),
  np.float64(163.6),
  np.float64(111.7),
  np.float64(137.5),
  np.float64(110.2),
  np.float64(132.7),
  np.float64(111.8),
  np.float64(118.9),
  np.float64(107.8),
  np.float64(128.2),
  np.float64(122.0),
  np.float64(102.1),
  np.float64(117.3),
  np.float64(148.4),
  np.float64(119.4),
  np.float64(127.5),
  np.float64(426.1),
  np.float64(127.2),
  np.float64(123.2),
  np.float64(116.4),
  np.float64(135.3),
  np.float64(172.8),
  np.float64(162.9),
  np.float64(128.3),
  np.float64(181.3),
  np.float64(139.4),
  np.float64(119.1),
  np.float64(212.7),
  np.float64(142.4),
  np.float64(186.9),
  np.float64(237.8),
  np.float64(168.0),
  np.float64(122.7),
  np.float64(116.7),
  np.float64(323.2),
  np.float64(146.1),
  np.float64(211.4),
  np.float64(95.8),
  np.float64(170.9),
  np.float64(124.8),
  np.float64(112.2),
  np.float64(223.9),
  np.float64(289.4),
  np.float64(131.1),
  np.float64(238.6),
  np.float64(109.1),
  np.float64(117.2),
  np.float64(122.3),
  np.float64(116.7),
  np.float64(123.3),
  np.float64(139.6),
  np.float64(115.9),
  np.float64(106.6),
  np.float64(108.4),
  np.float64(107.8),
  np.float64(113.3),
  np.float64(154.7),
  np.float64(352.7),
  np.float64(272.2),
  np.float64(137.6),
  np.float64(186.5),
  np.float64(163.4),
  np.float64(454.0),
  np.float64(109.3),
  np.float64(125.3),
  np.float64(126.8),
  np.float64(121.1),
  np.float64(268.1),
  np.float64(133.1),
  np.float64(132.7),
  np.float64(213.5),
  np.float64(134.2),
  np.float64(132.2),
  np.float64(142.8),
  np.float64(111.6),
  np.float64(209.3),
  np.float64(248.9),
  np.float64(260.3),
  np.float64(96.7),
  np.float64(113.6),
  np.float64(108.8),
  np.float64(121.9),
  np.float64(96.6),
  np.float64(106.0),
  np.float64(144.2),
  np.float64(102.6),
  np.float64(134.9),
  np.float64(136.6),
  np.float64(149.4),
  np.float64(111.3),
  np.float64(126.8),
  np.float64(126.8),
  np.float64(116.5),
  np.float64(132.9),
  np.float64(98.3),
  np.float64(174.7),
  np.float64(148.5),
  np.float64(122.6),
  np.float64(105.1),
  np.float64(106.7),
  np.float64(138.4),
  np.float64(135.9),
  np.float64(257.3),
  np.float64(157.1),
  np.float64(156.2),
  np.float64(206.4),
  np.float64(260.2),
  np.float64(165.1),
  np.float64(125.8),
  np.float64(341.6),
  np.float64(108.7),
  np.float64(136.1),
  np.float64(152.0),
  np.float64(106.3),
  np.float64(115.9),
  np.float64(188.4),
  np.float64(208.7),
  np.float64(112.4),
  np.float64(136.4),
  np.float64(183.2),
  np.float64(104.2),
  np.float64(107.8),
  np.float64(246.7),
  np.float64(228.9),
  np.float64(145.9),
  np.float64(129.5),
  np.float64(166.5),
  np.float64(170.0),
  np.float64(129.5),
  np.float64(184.6),
  np.float64(106.7),
  np.float64(331.7),
  np.float64(113.2),
  np.float64(115.0),
  np.float64(118.5),
  np.float64(368.5),
  np.float64(101.6),
  np.float64(328.0),
  np.float64(271.2),
  np.float64(113.7),
  np.float64(249.7),
  np.float64(101.2),
  np.float64(135.2),
  np.float64(94.8),
  np.float64(500.0)]

In [ ]:
data_paul = {"buffer_size": 1_000_000,
    "learning_starts": 100,
    "batch_size": 32,
    "tau": 1.0,
    "gamma": 0.99, # 0.99
    "train_freq": 1,
    "gradient_steps": 1,
    "replay_buffer_class": None,
    "replay_buffer_kwargs": None,
    "optimize_memory_usage": False,
    "n_steps": 1,
    "target_update_interval": 100, # 10_000
    "exploration_fraction": 0.4, # 0.1
    "exploration_initial_eps": 0.4,
    "exploration_final_eps": 0.001, # 0.05
    "max_grad_norm": 10.0}
test_hyperparameter(env_id =  'CartPole-v1',
                        seed = random.randint(0,1000000), 
                        timesteps_per_round  = 1000,
                        max_rounds = 1500,
                        reward_stop_count=1,
                        min_stop_value = 499,
                        evaluation_episodes = 10,
                        kwargs = data_paul)

In [ ]:
filename = "random_trial18.json"
print(filename)
with open("outputs/" + filename) as f:
    data = json.load(f)
test_hyperparameter(env_id =  'CartPole-v1',
                        seed = random.randint(0,1000000), 
                        timesteps_per_round  = 1000,
                        max_rounds = 1500,
                        reward_stop_count=1,
                        min_stop_value = 499,
                        evaluation_episodes = 10,
                        kwargs = data)